In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import requests
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
# #data source 1: list of universities and their domains
# url = "https://raw.githubusercontent.com/Hipo/university-domains-list/master/world_universities_and_domains.json"
# response = requests.get(url)
# universities_data = json.loads(response.text)
# universities_df = pd.DataFrame(universities_data)

# #filter to only include universities in the European Union
# eu_countries = [
#     "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
#     "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
#     "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
#     "Malta", "Netherlands", "Poland", "Portugal", "Romania",
#     "Slovakia", "Slovenia", "Spain", "Sweden"
# ]
# eu_universities_df = universities_df[universities_df['country'].isin(eu_countries)]
# eu_universities_df.reset_index(drop=True, inplace=True)

# #output the dataframe to a csv file
# eu_universities_df.to_csv('eu_universities.csv', index=False)

In [5]:
#data source 2: university budget statistics
df = pd.read_excel('../datafiles/University_budget_stats.xlsx', header=None, engine='openpyxl')
df = df[0].str.split(';', expand=True)
df.columns = df.iloc[0]
df = df.drop(0).reset_index(drop=True)
df.replace('m', pd.NA, inplace=True)

df.to_csv('../datafiles/University_bs.csv', index=False)
print(df.shape)
print(df.head(2))
df = df.rename(columns={
    'BAS.INSTNAME': 'name_local',
    'GEO.CITY': 'city',
    'REV.STUDFEES.EURO': 'student_fees',
    'STUD.HIGHDEG': 'highest_degree',
    'PERS.TOTALFTE': 'staff_fte',
    'EXP.CURRTOTAL.EURO': 'total_expenditure',
})
keeping_cols = ['student_fees', 'highest_degree', 'staff_fte']

model_df = df[["BAS.INSTNAMEENGL", 'city'] + keeping_cols].copy()
print("MODEL DF")
print(model_df.columns)
model_df = model_df.dropna(subset=keeping_cols).reset_index(drop=True)
model_df = model_df.rename(columns={
    'BAS.INSTNAMEENGL': 'name'
})

print(model_df.columns)

for col in keeping_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

model_df.reset_index(drop=True, inplace=True)

uni_df = pd.read_csv('../datafiles/eu_universities.csv')
uni_merge = uni_df[['name', 'web_pages']].drop_duplicates(subset='name')
model_df = pd.merge(model_df, uni_merge, on='name', how='left')
print(model_df.shape)
print(model_df[keeping_cols].describe())

(1000, 19)
0       BAS.INSTNAME      BAS.INSTNAMEENGL GEO.SUBCOUNTRY GEO.CITY  \
0  UniversitÃ¤t Wien  University of Vienna              a   Vienna   
1  UniversitÃ¤t Graz    University of Graz              a     Graz   

0 EXP.CURRPERSON.EURO EXP.CURRNONPERSON.EURO  EXP.CURRTOTAL.EURO  \
0  520378006.53000003     236841071.58999997        757219078.12   
1  212660260.34000003      90597029.80000001  303257290.14000005   

0 REV.COREBUDGETPUBLIC.EURO REV.CORETOTAL.EURO REV.THIRDPARTYPUBLIC.EURO  \
0              618382333.16       618382333.16                      <NA>   
1              255172805.74       255172805.74                      <NA>   

0 REV.THIRDPARTYPRIVATE.EURO REV.TUITFEES REV.STUDFEES.EURO PERS.ACAHCMEN  \
0                       <NA>            1       17131218.28          2517   
1                       <NA>            1        3628120.01          1085   

0 PERS.ACAHCWOMEN PERS.ACAHCSENIORTOTAL PERS.TOTALFTE STUD.HIGHDEG  \
0            2133                   557   

In [6]:
X_mat = model_df[keeping_cols].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_mat)
staff_min = model_df['staff_fte'].min()
staff_max = model_df['staff_fte'].max()

# # student survey inputs
# student_budget = 5000
# student_degree = 2
# student_size = 7

# # student survey inputs
# student_budget = 5000
# student_degree = 1
# student_size = 10

# student survey inputs
student_budget = 2000
student_degree = 3
student_size = 2

student_staff = staff_min + (student_size - 1) * (staff_max - staff_min) / 9

student_input = np.array([[student_budget, student_degree, student_staff]])
student_scaled = scaler.transform(student_input)

cosine_scores = []

for i in range(X_scaled.shape[0]):
    uni_vec = X_scaled[i]
    student_vec = student_scaled[0]
    dot = np.dot(student_vec, uni_vec)
    cos = dot / (np.linalg.norm(student_vec) * np.linalg.norm(uni_vec))
    cosine_scores.append(cos)

results_df = pd.DataFrame({
    'name':         model_df['name'],
    'city':         model_df['city'],
    'cosine_score': cosine_scores
})

ranked = results_df.sort_values(by='cosine_score', ascending=False)
ranked = ranked.reset_index(drop=True)
ranked.index = ranked.index + 1
ranked['match_number'] = (ranked['cosine_score'] * 100).round(2).astype(str)

output = ranked[['name', 'city', 'match_number']]
output.index.name = 'rank'

print("TOP 10 MATCHES")
print(output.head(10).to_string())
print("\n... view more ...")
print(output.tail().to_string())

TOP 10 MATCHES
                                               name        city match_number
rank                                                                        
1     Technische UniversitÃ¤t Bergakademie Freiberg    Freiberg        99.97
2                  Ilmenau University of Technology     Ilmenau        99.87
3            Hamburg University of Applied Sciences     Hamburg         99.6
4                             OsnabrÃ¼ck University  OsnabrÃ¼ck        99.48
5                             University of Bamberg     Bamberg        99.43
6                          University of Greifswald  Greifswald         99.4
7                               University of Trier       Trier        99.34
8          University of Veterinary Medicine Vienna      Vienna        99.04
9                            University of Mannheim    Mannheim        99.03
10                 Hamburg University of Technology     Hamburg         98.9

... view more ...
                                          